## Lokalizacja punktu w przestrzeni dwuwymiarowej – metoda separatorów

### Cezary Szczepanek, Patryk Opala

In [1]:
import numpy as np
from graph import *


In [2]:
g = Graph()
            #0      #1      #2      #3
vertices = [(0, 0), (1, 1), (2, 2), (3, 2)]

for x, y in vertices:
    g.add_vertex(x, y)

edges = [(0, 1), (1, 2), (2, 3), (3, 0), (1, 3)]

for ver1, ver2 in edges:
    g.add_edge(ver1, ver2)

ver = g.vertices.items()
print(ver)

for v in ver:
    print(v[1].in_edges)
    print(v[1].out_edges)
    print()

dict_items([(0, V0(0, 0)), (1, V1(1, 1)), (2, V2(2, 2)), (3, V3(3, 2))])
[]
[Edge(0 -> 1, w=0), Edge(0 -> 3, w=0)]

[Edge(0 -> 1, w=0)]
[Edge(1 -> 2, w=0), Edge(1 -> 3, w=0)]

[Edge(1 -> 2, w=0)]
[Edge(2 -> 3, w=0)]

[Edge(2 -> 3, w=0), Edge(0 -> 3, w=0), Edge(1 -> 3, w=0)]
[]



In [3]:
for edge in g.edges:
    edge.weight = 1

sorted_v = g.get_sorted_vertices()
n = len(sorted_v)
for i in range(0, n - 1):
    v = sorted_v[i]

    if v.total_in_weight > v.total_out_weight:
        out_edges = g.get_sorted_out_edges(v)
        leftmost_edge = out_edges[0] 
        leftmost_edge.weight = v.total_in_weight - v.total_out_weight + leftmost_edge.weight


sorted_v_reverse = list(reversed(sorted_v))
for i in range(0, n - 1):
    v = sorted_v_reverse[i]

    if v.total_out_weight > v.total_in_weight:
        in_edges = g.get_sorted_in_edges(v)
        leftmost_edge = in_edges[0]
        leftmost_edge.weight = v.total_out_weight - v.total_in_weight + leftmost_edge.weight


In [4]:
def create_chain(g: Graph):
    sorted_v = g.get_sorted_vertices()
    num_chains = sorted_v[0].total_out_weight
    chains = [[] for _ in range(num_chains)]

    for chain in range(num_chains):
        v = sorted_v[0]
        while v != sorted_v[-1]:
            edges = g.get_sorted_out_edges(v)
            selected_edge = None

            for edge in edges:
                if edge.weight != 0:
                    edge.weight -= 1
                    edge.chains.append(chain)
                    selected_edge = edge
                    break
            
            if selected_edge:
                chains[chain].append(selected_edge)
                v = selected_edge.end
            else:
                print("Nie zbalansowany graf")
                return 
                

    return chains            
            


chains = create_chain(g)
print(chains)

[[Edge(0 -> 3, w=0)], [Edge(0 -> 1, w=0), Edge(1 -> 3, w=0)], [Edge(0 -> 1, w=0), Edge(1 -> 2, w=0), Edge(2 -> 3, w=0)]]


In [6]:
def find_edge_in_chain(y: float, chain: List[Edge]) -> Optional[Edge]:
    low = 0
    high = len(chain) - 1
    
    while low <= high:
        mid = (low + high) // 2
        edge = chain[mid]
        
        # Sprawdzamy czy punkt y mieści się w przedziale y krawędzi
        if edge.start.y <= y <= edge.end.y:
            return edge
        elif y < edge.start.y:
            high = mid - 1
        else:
            low = mid + 1
    return None

def is_left_of_edge(p_x: float, p_y: float, edge: Edge) -> bool:
    """
    Zwraca True, jeśli punkt (p_x, p_y) leży po lewej stronie krawędzi 'edge'.
    Krawędź jest skierowana od edge.start do edge.end.
    """
    # Wektor krawędzi (v1) i wektor do punktu (v2)
    v1_x, v1_y = edge.end.x - edge.start.x, edge.end.y - edge.start.y
    v2_x, v2_y = p_x - edge.start.x, p_y - edge.start.y
    
    # Iloczyn wektorowy 2D
    det = v1_x * v2_y - v1_y * v2_x
    
    # UWAGA: W standardowym układzie współrzędnych (Y rośnie w górę):
    # det > 0 oznacza, że punkt jest po LEWEJ stronie.
    return det > 0


def locate_point(px: float, py: float, chains: List[List[Edge]]):
    low = 0
    high = len(chains) - 1
    ans_left = -1
    ans_right = len(chains)

    while low <= high:
        mid = (low + high) // 2
        chain = chains[mid]
        
        # 1. Znajdź krawędź na tej samej wysokości co punkt
        edge = find_edge_in_chain(py, chain)
        
        if edge is None:
            # Punkt jest poza zakresem Y tego łańcucha
            return "Poza zakresem Y grafu"

        # 2. Porównaj punkt z krawędzią
        if is_left_of_edge(px, py, edge):
            # Punkt jest na lewo od łańcucha, szukaj w lewiejszych
            ans_right = mid
            high = mid - 1
        else:
            # Punkt jest na prawo od łańcucha, szukaj w prawiejszych
            ans_left = mid
            low = mid + 1

    return f"Punkt znajduje się między łańcuchem {ans_left} a {ans_right}"

In [15]:
print(locate_point(3, 1.5, chains))

Punkt znajduje się między łańcuchem 2 a 3
